[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/legalintermedia/Physics/blob/main/Simulaciones_Jupyter/Investigacion_Avanzada/Termodinamica_MBL.ipynb)



# Investigación Avanzada: Termodinamica MBL

Simulación numérica de vanguardia correspondiente a los Problemas Abiertos y Fronteras de Investigación (Nivel Doctorado).

In [ ]:
%matplotlib inline
import numpy as np
import scipy.sparse as sp
import scipy.sparse.linalg as spla
import matplotlib.pyplot as plt

def heisenberg_hamiltonian_1d(L, W, J=1.0):
    """
    Constructs the Hamiltonian for a 1D spin-1/2 Heisenberg chain with random onsite disorder.
    H = J \sum (S^x_i S^x_{i+1} + S^y_i S^y_{i+1} + S^z_i S^z_{i+1}) + \sum h_i S^z_i
    where h_i in [-W, W].
    """
    dim = 2**L
    
    # Pauli matrices
    sz = np.array([[1, 0], [0, -1]])
    sp_op = np.array([[0, 1], [0, 0]]) # S+
    sm_op = np.array([[0, 0], [1, 0]]) # S-
    
    def kronecker_pad(op, pos):
        """ Place operator at pos in the tensor product space """
        res = 1
        for i in range(L):
            if i == pos:
                res = sp.kron(res, op, format='csr') if not isinstance(res, int) else sp.csr_matrix(op)
            else:
                eye = sp.eye(2)
                res = sp.kron(res, eye, format='csr') if not isinstance(res, int) else eye
        return res
    
    H = sp.csr_matrix((dim, dim), dtype=np.float64)
    
    # Random fields
    fields = np.random.uniform(-W, W, L)
    
    for i in range(L):
        # On-site disorder
        Sz_i = kronecker_pad(sz, i)
        H += 0.5 * fields[i] * Sz_i
        
        # Interaction with next neighbor (periodic boundary conditions)
        j = (i + 1) % L
        Sz_j = kronecker_pad(sz, j)
        Sp_i = kronecker_pad(sp_op, i)
        Sm_i = kronecker_pad(sm_op, i)
        Sp_j = kronecker_pad(sp_op, j)
        Sm_j = kronecker_pad(sm_op, j)
        
        # S_i . S_j = Sz_i Sz_j + 0.5 * (Sp_i Sm_j + Sm_i Sp_j)
        H += J * (0.25 * Sz_i.dot(Sz_j) + 0.5 * 0.5 * (Sp_i.dot(Sm_j) + Sm_i.dot(Sp_j)))
        
    return H

def calculate_level_spacing(L, W_list, num_realizations=5):
    r_mean = []
    
    for W in W_list:
        r_vals = []
        for _ in range(num_realizations):
            H = heisenberg_hamiltonian_1d(L, W)
            # Find eigenvalues in the middle of the spectrum
            evals, evecs = spla.eigsh(H, k=20, which='SM')
            evals = np.sort(evals)
            
            # Level spacings
            spacings = np.diff(evals)
            r = np.minimum(spacings[:-1], spacings[1:]) / np.maximum(spacings[:-1], spacings[1:])
            r_vals.extend(r)
            
        r_mean.append(np.mean(r_vals))
        print(f"W={W}, <r> = {r_mean[-1]:.4f}")
        
    return r_mean

def simulate_mbl():
    L = 10 # System size (keep small for Exact Diagonalization)
    W_list = np.linspace(0.5, 8.0, 10)
    
    r_mean = calculate_level_spacing(L, W_list, num_realizations=10)
    
    plt.figure(figsize=(8, 6))
    plt.plot(W_list, r_mean, 'bo-', lw=2)
    plt.axhline(0.5307, color='r', linestyle='--', label='GOE (Ergodic)')
    plt.axhline(0.386, color='g', linestyle='--', label='Poisson (Localized)')
    plt.xlabel('Disorder Strength $W$')
    plt.ylabel('Mean Level Spacing Ratio $\langle r \\rangle$')
    plt.title('Many-Body Localization (MBL) Transition in 1D Heisenberg Model')
    plt.legend()
    plt.grid(True)
    plt.savefig('MBL_Transition.png', dpi=300)
    print("MBL simulation complete. Image saved.")

if __name__ == '__main__':
    simulate_mbl()
